In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def best_params_by_phase(metric, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range):
    rows = []
    for pi, phase_diff in enumerate(phase_diff_range):
        slab = metric[:, :, :, :, pi]                       # fix phase, keep the other 4 axes
        fh_i, fg_i, dh_i, dg_i = np.unravel_index(np.argmax(slab), slab.shape)
        rows.append({'freq_h': freq_h_range[fh_i], 'freq_g': freq_g_range[fg_i],
                     'duty_h': duty_h_range[dh_i], 'duty_g': duty_g_range[dg_i],
                     'phase_diff': phase_diff, 'score': float(slab[fh_i, fg_i, dh_i, dg_i])})
    return rows

def plot_freqs(frac_grid, frac_hc, duty_g, duty_h, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range):
    duty_h_idx = np.where(np.isclose(duty_h_range, duty_h))[0][0]
    duty_g_idx = np.where(np.isclose(duty_g_range, duty_g))[0][0]

    print(f"Fixed duty_h at {duty_h_range[duty_h_idx]:.1f} and duty_g at {duty_g_range[duty_g_idx]:.1f}")


    grid_for_fixed_duties = frac_grid[:, :, duty_h_idx, duty_g_idx, :]
    hc_for_fixed_duties = frac_hc[:, :, duty_h_idx, duty_g_idx, :]

    num_phases = len(phase_diff_range)

    fig, axes = plt.subplots(num_phases, 2, figsize=(12, 4 * num_phases), squeeze=False)

    for pd_idx, phase_diff_val in enumerate(phase_diff_range):
        grid_heatmap_data = grid_for_fixed_duties[:, :, pd_idx]
        hc_heatmap_data = hc_for_fixed_duties[:, :, pd_idx]

        # Plot for Grid Recovery
        im0 = axes[pd_idx, 0].imshow(grid_heatmap_data, origin='lower', vmin=0, vmax=1,
                                    extent=[freq_g_range[0], freq_g_range[-1],
                                            freq_h_range[0], freq_h_range[-1]],
                                    aspect='auto', cmap='viridis')
        axes[pd_idx, 0].set_title(f'Grid Rec. (Phase diff={phase_diff_val:.1f}, Dh={duty_h_range[duty_h_idx]:.1f}, Dg={duty_g_range[duty_g_idx]:.1f})')
        axes[pd_idx, 0].set_xlabel('Freq_g')
        axes[pd_idx, 0].set_ylabel('Freq_h')
        fig.colorbar(im0, ax=axes[pd_idx, 0], shrink=0.7, label='Grid Recovery')

        # Plot for HC Recovery
        im1 = axes[pd_idx, 1].imshow(hc_heatmap_data, origin='lower', vmin=0, vmax=1,
                                    extent=[freq_g_range[0], freq_g_range[-1],
                                            freq_h_range[0], freq_h_range[-1]],
                                    aspect='auto', cmap='viridis')
        axes[pd_idx, 1].set_title(f'HC Rec. (Phase diff={phase_diff_val:.1f}, Dh={duty_h_range[duty_h_idx]:.1f}, Dg={duty_g_range[duty_g_idx]:.1f})')
        axes[pd_idx, 1].set_xlabel('Freq_g')
        axes[pd_idx, 1].set_ylabel('Freq_h')
        fig.colorbar(im1, ax=axes[pd_idx, 1], shrink=0.7, label='HC Recovery')

    plt.tight_layout()
    plt.show()

def plot_duties(frac_grid, frac_hc, f_common_idx, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range):
    f_common_idx = 0
    f_common_val = freq_h_range[f_common_idx]

    print(f"Generating heatmaps for common frequency: {f_common_val} Hz")

    # frac_grid[f_common_idx, f_common_idx, :, :, :] will give (Dh, Dg, Pd)
    grid_for_fixed_freq = frac_grid[f_common_idx, f_common_idx, :, :, :]
    hc_for_fixed_freq = frac_hc[f_common_idx, f_common_idx, :, :, :]

    num_phases = len(phase_diff_range)

    fig, axes = plt.subplots(num_phases, 2, figsize=(12, 4 * num_phases), squeeze=False)

    for pd_idx, phase_diff_val in enumerate(phase_diff_range):
            grid_heatmap_data = grid_for_fixed_freq[:, :, pd_idx]
            hc_heatmap_data = hc_for_fixed_freq[:, :, pd_idx]

            # Plot for Grid Recovery
            im0 = axes[pd_idx, 0].imshow(grid_heatmap_data, origin='lower', vmin=0, vmax=1,
                                            extent=[duty_g_range[0], duty_g_range[-1],
                                                    duty_h_range[0], duty_h_range[-1]],
                                            aspect='auto', cmap='viridis')
            axes[pd_idx, 0].set_title(f'Grid Rec. (Phase diff={phase_diff_val:.1f}, Freq={f_common_val:.2f})')
            axes[pd_idx, 0].set_xlabel('Duty_g')
            axes[pd_idx, 0].set_ylabel('Duty_h')
            fig.colorbar(im0, ax=axes[pd_idx, 0], shrink=0.7, label='Grid Recovery')

            # Plot for HC Recovery
            im1 = axes[pd_idx, 1].imshow(hc_heatmap_data, origin='lower', vmin=0, vmax=1,
                                            extent=[duty_g_range[0], duty_g_range[-1],
                                                    duty_h_range[0], duty_h_range[-1]],
                                            aspect='auto', cmap='viridis')
            axes[pd_idx, 1].set_title(f'HC Rec. (Phase diff={phase_diff_val:.1f}, Freq={f_common_val:.2f})')
            axes[pd_idx, 1].set_xlabel('Duty_g')
            axes[pd_idx, 1].set_ylabel('Duty_h')
            fig.colorbar(im1, ax=axes[pd_idx, 1], shrink=0.7, label='HC Recovery')

    plt.tight_layout()
    plt.show()
    

# Multiplicative theta

## e = 5, i = -8.75

In [ ]:
frac_grid = np.load('/home/srujana/VSCode Projects/Thesis/vectorHASH-cont-jax/parameter_search_results/2 theta/mulitplicative waves/e5_i875/frac_grid_2thetas.npy')
frac_hc = np.load('/home/srujana/VSCode Projects/Thesis/vectorHASH-cont-jax/parameter_search_results/2 theta/mulitplicative waves/e5_i875/frac_hc_2thetas.npy')

freq_h_range = np.arange(0.5, 4.0, 0.5)
freq_g_range = np.arange(0.5, 4.0, 0.5)
duty_h_range = np.arange(0.0, 1.1, 0.1)
duty_g_range = np.arange(0.0, 1.1, 0.1)
phase_diff_range = np.arange(0.0, 1.1, 0.1)
phase_h = 0.0

In [ ]:
rows = best_params_by_phase(frac_grid, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range)
for row in sorted(rows, key=lambda x: x['score'], reverse=True):
    clean_row = {k: f'{v:.3f}' if isinstance(v, (np.float64, float)) else v for k, v in row.items()}
    print(clean_row)

In [ ]:
plot_duties(frac_grid, frac_hc, 0, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range)

In [ ]:
plot_freqs(frac_grid, frac_hc, duty_g, duty_h, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range)

## e = 5, i = -7

In [2]:
frac_grid = np.load("/home/srujana/VSCode Projects/Thesis/vectorHASH-cont-jax/parameter_search_results/2 theta/mulitplicative waves/e5_i7/frac_grid_e5_i7.npy")
frac_hc = np.load("/home/srujana/VSCode Projects/Thesis/vectorHASH-cont-jax/parameter_search_results/2 theta/mulitplicative waves/e5_i7/frac_hc_e5_i7.npy")

print(frac_grid.shape)
freq_h_range = np.arange(0.5, 4.0, 0.5)
freq_g_range = np.arange(0.5, 4.0, 0.5)
duty_h_range = np.arange(0.0, 1.1, 0.1)
duty_g_range = np.arange(0.0, 1.1, 0.1)
phase_diff_range = np.arange(0.0, 1.1, 0.1)
phase_h = 0.0

(7, 7, 11, 11, 11)


In [ ]:
rows = best_params_by_phase(frac_grid, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range)
for row in sorted(rows, key=lambda x: x['score'], reverse=True):
    clean_row = {k: f'{v:.3f}' if isinstance(v, (np.float64, float)) else v for k, v in row.items()}
    print(clean_row)

In [ ]:
plot_duties(frac_grid, frac_hc, 0, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range)

In [ ]:
duty_g, duty_h = 0.5, 0.5
plot_freqs(frac_grid, frac_hc, duty_g, duty_h, freq_h_range, freq_g_range, duty_h_range, duty_g_range, phase_diff_range)